# Therapeutic Alignment Evaluation - SAFE TEST VERSION

This is a **conservative test version** with reduced concurrency to avoid overwhelming Ollama.

## Test Configuration:
- **Files**: 1 file only (for testing)
- **Turns**: 20 turns per file (limited)
- **File Workers**: 1 (sequential files)
- **Turn Workers**: 3 (low concurrency)
- **Delay**: 0.3s between calls

## Expected Time:
- ~5-10 minutes for 1 file with 20 turns

If this works without errors, you can gradually increase the settings.

## 1. Setup and Installation

In [1]:
import sys
import os
import json
from pathlib import Path
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import time
from dataclasses import asdict
from tqdm.auto import tqdm

# Add our-pipeline to path
pipeline_path = Path("./our-pipeline")
if str(pipeline_path) not in sys.path:
    sys.path.insert(0, str(pipeline_path))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import existing modules
from transcript_parser import (
    parse_html_transcript_file,
    get_counselor_turns,
    get_patient_turns,
    get_conversation_context
)
from alignment_evaluators import (
    create_openai_client,
    create_ollama_client,
    create_lmstudio_client,
    evaluate_cbt_adherence_with_memory,
    evaluate_persona_consistency_with_memory,
    calculate_statistics
)

# Import Mem0 integration
from mem0_integration import (
    initialize_mem0,
    create_mem0_config_with_llm,
    add_conversation_turn_to_memory,
    get_all_memories,
    get_memory_at_turn,
    audit_memories,
    calculate_memory_statistics,
    format_memories_for_audit
)

print("✅ All modules loaded successfully!")

✅ All modules loaded successfully!


## 2. Model Configuration

In [2]:
# Using Ollama with gpt-oss:20b
USE_OLLAMA = True
OLLAMA_MODEL = "gpt-oss:20b"

client = create_ollama_client()
MODEL = OLLAMA_MODEL

print(f"✅ Using Ollama with model: {MODEL}")
print("⚠️  Make sure Ollama is running: ollama serve")

✅ Using Ollama with model: gpt-oss:20b
⚠️  Make sure Ollama is running: ollama serve


## 3. Initialize Mem0

In [3]:
RESET_MEMORIES = True  # Start fresh for testing

mem_config = create_mem0_config_with_llm(
    llm_provider="ollama",
    model=OLLAMA_MODEL,
    base_url="http://localhost:11434"
)
mem_config["vector_store"]["config"]["collection_name"] = "therapy_memories_safe_test"

memory = initialize_mem0(
    config=mem_config,
    reset_collection=RESET_MEMORIES
)

print(f"✅ Mem0 initialized with Ollama LLM: {OLLAMA_MODEL}")
print(f"   Collection: therapy_memories_safe_test")
print(f"   Path: ./chroma_db")

Resetting index therapy_memories_safe_test...


✅ Mem0 initialized with Ollama LLM: gpt-oss:20b
   Collection: therapy_memories_safe_test
   Path: ./chroma_db


## 4. Load Transcript Files (LIMITED TO 1 FILE)

In [4]:
# Load transcript files
DATASET_DIR = Path("./0518-014_raw")
all_transcript_files = sorted(list(DATASET_DIR.glob("*.txt")))

# LIMIT TO FIRST FILE FOR TESTING
transcript_files = all_transcript_files[:1]

print(f"📁 Found {len(all_transcript_files)} total files")
print(f"🧪 Testing with {len(transcript_files)} file(s)")
print("=" * 60)

print("\nFile to process:")
for idx, file in enumerate(transcript_files, 1):
    print(f"  {idx}. {file.name}")

📁 Found 16 total files
🧪 Testing with 1 file(s)

File to process:
  1. 1000056544.txt


## 5. SAFE Configuration for Testing

In [5]:
# ============================================================================
# SAFE TEST CONFIGURATION
# ============================================================================

MAX_TURNS = 20  # Only process first 20 turns per file

# Conservative parallelization settings
MAX_FILE_WORKERS = 1  # Process 1 file at a time
MAX_TURN_WORKERS = 3  # Evaluate 3 turns concurrently

# Add delay to avoid overwhelming Ollama
DELAY_BETWEEN_CALLS = 0.3  # 300ms delay between calls

# Create output directory for results
OUTPUT_DIR = Path("./evaluation_results_safe_test")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"🔧 Configuration:")
print(f"   Model: {MODEL}")
print(f"   Max file workers: {MAX_FILE_WORKERS}")
print(f"   Max turn workers: {MAX_TURN_WORKERS}")
print(f"   Delay between calls: {DELAY_BETWEEN_CALLS}s")
print(f"   Max turns per file: {MAX_TURNS}")
print(f"   Total files: {len(transcript_files)}")
print(f"   Output directory: {OUTPUT_DIR}")
print("=" * 60)

🔧 Configuration:
   Model: gpt-oss:20b
   Max file workers: 1
   Max turn workers: 3
   Delay between calls: 0.3s
   Max turns per file: 20
   Total files: 1
   Output directory: evaluation_results_safe_test


## 6. Processing with Safe Settings

In [6]:
# Thread-safe lock for memory operations
memory_lock = Lock()

def evaluate_single_turn(turn, turns, baseline_response, USER_ID):
    """Evaluate a single counselor turn (thread-safe)."""
    try:
        # Get conversation context
        context = get_conversation_context(turns, turn.turn_number, max_turns=10)
        
        # Get memories up to this turn (thread-safe)
        with memory_lock:
            memories_up_to_turn = get_memory_at_turn(
                memory=memory,
                turn_number=turn.turn_number,
                user_id=USER_ID
            )
        
        # Format memories for evaluation
        memories_formatted = format_memories_for_audit(memories_up_to_turn)
        
        # Evaluate CBT adherence with memory context
        cbt_result = evaluate_cbt_adherence_with_memory(
            client=client,
            counselor_response=turn.content,
            conversation_context=context,
            memories_context=memories_formatted,
            turn_number=turn.turn_number,
            model=MODEL
        )
        
        time.sleep(DELAY_BETWEEN_CALLS)
        
        # Evaluate persona consistency with memory context
        persona_result = evaluate_persona_consistency_with_memory(
            client=client,
            counselor_response=turn.content,
            baseline_response=baseline_response,
            conversation_context=context,
            memories_context=memories_formatted,
            turn_number=turn.turn_number,
            model=MODEL
        )
        
        time.sleep(DELAY_BETWEEN_CALLS)
        
        # Get current memory count (thread-safe)
        with memory_lock:
            current_memories = get_all_memories(memory, USER_ID)
        
        return {
            'turn_number': turn.turn_number,
            'cbt_result': asdict(cbt_result),
            'persona_result': asdict(persona_result),
            'memory_snapshot': {
                "turn_number": turn.turn_number,
                "memory_count": len(current_memories),
                "memories_in_context": len(memories_up_to_turn),
                "cbt_score": cbt_result.score,
                "persona_score": persona_result.score
            }
        }
    except Exception as e:
        print(f"❌ Error evaluating turn {turn.turn_number}: {e}")
        return None

def process_single_file(transcript_file):
    """Process a single transcript file."""
    try:
        print(f"\n📄 Processing: {transcript_file.name}")
        
        # Use filename as unique USER_ID for memory isolation
        USER_ID = f"session_{transcript_file.stem}"
        
        # Parse the transcript
        turns = parse_html_transcript_file(str(transcript_file))
        print(f"   Loaded {len(turns)} total turns")
        
        # Get counselor turns for evaluation
        counselor_turns = get_counselor_turns(turns)
        if MAX_TURNS:
            counselor_turns = counselor_turns[:MAX_TURNS]
        
        print(f"   Will evaluate {len(counselor_turns)} counselor turns")
        
        # Store baseline for persona comparison
        baseline_response = counselor_turns[0].content if counselor_turns else ""
        
        # Add all turns to memory first (sequential to maintain order)
        print(f"\n🧠 Adding turns to memory...")
        for i, turn in enumerate(turns):
            if MAX_TURNS and turn.turn_number > MAX_TURNS * 2:
                break
            
            if i % 10 == 0:
                print(f"   Adding turn {i+1}/{min(len(turns), MAX_TURNS*2)}...")
            
            with memory_lock:
                add_conversation_turn_to_memory(
                    memory=memory,
                    turn_content=turn.content,
                    role=turn.role,
                    turn_number=turn.turn_number,
                    user_id=USER_ID
                )
        
        print(f"   ✅ Memory addition complete")
        
        # Now evaluate counselor turns with limited concurrency
        print(f"\n📊 Evaluating counselor turns...")
        results = []
        
        with ThreadPoolExecutor(max_workers=MAX_TURN_WORKERS) as executor:
            # Submit all evaluation tasks
            future_to_turn = {
                executor.submit(evaluate_single_turn, turn, turns, baseline_response, USER_ID): turn
                for turn in counselor_turns
            }
            
            # Collect results with progress bar
            with tqdm(total=len(counselor_turns), desc=f"Evaluating {transcript_file.name}") as pbar:
                for future in as_completed(future_to_turn):
                    result = future.result()
                    if result:
                        results.append(result)
                    pbar.update(1)
        
        # Sort results by turn number
        results.sort(key=lambda x: x['turn_number'])
        
        # Extract organized results
        cbt_results = [r['cbt_result'] for r in results]
        persona_results = [r['persona_result'] for r in results]
        memory_snapshots = [r['memory_snapshot'] for r in results]
        
        # Save results to JSON file
        output_filename = transcript_file.stem + "_safe_test.json"
        output_path = OUTPUT_DIR / output_filename
        
        results_data = {
            "filename": transcript_file.name,
            "total_turns": len(turns),
            "counselor_turns_evaluated": len(cbt_results),
            "model": MODEL,
            "test_mode": True,
            "max_turns": MAX_TURNS,
            "max_turn_workers": MAX_TURN_WORKERS,
            "delay_between_calls": DELAY_BETWEEN_CALLS,
            "cbt_adherence_results": cbt_results,
            "persona_consistency_results": persona_results,
            "memory_snapshots": memory_snapshots
        }
        
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results_data, f, indent=2, ensure_ascii=False)
        
        print(f"\n✅ Saved results to: {output_path}")
        
        return {
            'filename': transcript_file.name,
            'output_path': str(output_path),
            'turns_evaluated': len(cbt_results)
        }
        
    except Exception as e:
        print(f"❌ Error processing {transcript_file.name}: {e}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# MAIN PROCESSING
# ============================================================================

print("\n🚀 Starting SAFE TEST processing...")
print("=" * 60)

start_time = time.time()
completed_files = []

# Process file(s)
for file in transcript_files:
    result = process_single_file(file)
    if result:
        completed_files.append(result)

end_time = time.time()
elapsed_time = end_time - start_time

print("\n" + "=" * 60)
print(f"✅ SAFE TEST COMPLETED!")
print(f"⏱️  Total time: {elapsed_time/60:.2f} minutes ({elapsed_time:.2f} seconds)")
print(f"📁 Files completed: {len(completed_files)}/{len(transcript_files)}")
print(f"💾 Results saved to: {OUTPUT_DIR}/")
print("=" * 60)

if len(completed_files) == len(transcript_files):
    print("\n✅ SUCCESS! No errors detected.")
    print("\n💡 Next steps:")
    print("   1. Review the results in the output directory")
    print("   2. If successful, you can increase MAX_TURN_WORKERS to 4-6")
    print("   3. Then increase MAX_TURNS to process more turns")
    print("   4. Finally, process more files by changing transcript_files[:1] to [:2], etc.")
else:
    print("\n⚠️  Some files failed. Check the error messages above.")


🚀 Starting SAFE TEST processing...

📄 Processing: 1000056544.txt
   Loaded 250 total turns
   Will evaluate 20 counselor turns

🧠 Adding turns to memory...
   Adding turn 1/40...
   Adding turn 11/40...
   Adding turn 21/40...
   Adding turn 31/40...


Empty response from LLM, no memories to extract


   ✅ Memory addition complete

📊 Evaluating counselor turns...


Evaluating 1000056544.txt:   0%|          | 0/20 [00:00<?, ?it/s]

    "score": 3,
    "linguistic_distance": 0.9,
    "professional_indicators": ["maintains boundaries"],
    "drift_indicators": ["informal tone", "peer-like language", "mirrors client language"],
 ...

✅ Saved results to: evaluation_results_safe_test\1000056544_safe_test.json

✅ SAFE TEST COMPLETED!
⏱️  Total time: 31.18 minutes (1870.53 seconds)
📁 Files completed: 1/1
💾 Results saved to: evaluation_results_safe_test/

✅ SUCCESS! No errors detected.

💡 Next steps:
   1. Review the results in the output directory
   2. If successful, you can increase MAX_TURN_WORKERS to 4-6
   3. Then increase MAX_TURNS to process more turns
   4. Finally, process more files by changing transcript_files[:1] to [:2], etc.


## 7. View Results Summary

In [7]:
# Load and display results
result_files = list(OUTPUT_DIR.glob("*_safe_test.json"))

if result_files:
    print(f"📊 Results Summary")
    print("=" * 60)
    
    for result_file in result_files:
        with open(result_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"\nFile: {data['filename']}")
        print(f"  Turns evaluated: {data['counselor_turns_evaluated']}")
        
        cbt_scores = [r['score'] for r in data['cbt_adherence_results']]
        persona_scores = [r['score'] for r in data['persona_consistency_results']]
        
        print(f"  CBT Adherence:")
        print(f"    Mean: {sum(cbt_scores)/len(cbt_scores):.2f}/10")
        print(f"    Range: {min(cbt_scores)}-{max(cbt_scores)}")
        
        print(f"  Persona Consistency:")
        print(f"    Mean: {sum(persona_scores)/len(persona_scores):.2f}/10")
        print(f"    Range: {min(persona_scores)}-{max(persona_scores)}")
        
        print(f"  Memories extracted: {data['memory_snapshots'][-1]['memory_count']}")
else:
    print("❌ No result files found. Check if processing completed successfully.")

📊 Results Summary

File: 1000056544.txt
  Turns evaluated: 20
  CBT Adherence:
    Mean: 2.25/10
    Range: 1-3
  Persona Consistency:
    Mean: 5.65/10
    Range: 1-10
  Memories extracted: 35
